# BINARY FEATURES: Flag, Bool, True-False

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
!pip install missingno
import missingno as msno
from datetime import date
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor # çok değişkenli local outlier yaklama yöntemidir.
from sklearn.preprocessing import MinMaxScaler, LabelEncoder, StandardScaler, RobustScaler

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)
pd.set_option('display.width', 500)

In [2]:
def load():
    df = pd.read_csv("titanic.csv")
    return df

In [3]:
#BU konuda genel kabul gören bir literatür yokur çünkü işlemden işleme değişmektedir.
#burada yapacak olduğumuz işlem 1 ve 0 üzerinden yeni değişkenler vermek olacak, 
df = load()

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.000,1,0,A/5 21171,7.250,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.000,1,0,PC 17599,71.283,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.000,0,0,STON/O2. 3101282,7.925,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.000,1,0,113803,53.100,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.000,0,0,373450,8.050,NaN,S


In [4]:
#öyle bir işlem yapmak istiyorum ki nan gördüğüm yerlere 1 nan olmayan yerlere 0 yazmak 
# istiyorum. normlade kabin değişkeni çöp bir değişken çünkü fazla sayıda nan değeri var.
#ancak ben bunları 1 ile doldurup bi bakayım diyorum gerçekten anlamsız bir değişken mi 
# bağımlı değişken ile aralarında hiçbir bağlantı var mı diyorum.

df["NEW_CABIN_BOOL"] = df["Cabin"].notnull().astype("int")
df #buardaki isimlendirmeye flag de yazabiliriz isimlendirme istediğimiz gibi olabilir

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,NEW_CABIN_BOOL
0,1,0,3,"Braund, Mr. Owen Harris",male,22.000,1,0,A/5 21171,7.250,NaN,S,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.000,1,0,PC 17599,71.283,C85,C,1
2,3,1,3,"Heikkinen, Miss. Laina",female,26.000,0,0,STON/O2. 3101282,7.925,NaN,S,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.000,1,0,113803,53.100,C123,S,1
4,5,0,3,"Allen, Mr. William Henry",male,35.000,0,0,373450,8.050,NaN,S,0
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.458,NaN,Q,0
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.000,0,0,17463,51.862,E46,S,1
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.000,3,1,349909,21.075,NaN,S,0
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.000,0,2,347742,11.133,NaN,S,0
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.000,1,0,237736,30.071,NaN,C,0


In [5]:
#şimdi bağımlı değişkenim surviveda göre ortalama alacağım

df.groupby("NEW_CABIN_BOOL").agg({"Survived": "mean"})

# dikkat kabin numarası dolu olanların hayatta kalma oranı kabini boş olanların hayatta kalma oranına göre daha yüksek
#benim için başta çöp olan değişken aslında şu an benim için en önemli değişkenlerden biri oldu.

,Survived
NEW_CABIN_BOOL,
0,0.300
1,0.667


In [7]:
#şimdi burada istatistiki bir test yapacağım, iki grubun oranını kıyaslamak için
#bu değişkenler birbiri ile ilişkili mi değil mi diye, elimde yeni oluşturduğum bir değişken var
#ve bu değişkenin bağımlı değişken ile bir ilişkisi var mı bunu merak ediyorum.

#bunun için oran testi yapıyorum.

from statsmodels.stats.proportion import proportions_ztest

test_stat, pvalue = proportions_ztest(count=[df.loc[df["NEW_CABIN_BOOL"] == 1, "Survived"].sum(),
                                            df.loc[df["NEW_CABIN_BOOL"] == 0, "Survived"].sum()],
                                      
                                      nobs=[df.loc[df["NEW_CABIN_BOOL"] == 1, "Survived"].shape[0],
                                           df.loc[df["NEW_CABIN_BOOL"] == 0, "Survived"].shape[0]])

print('Test Stat = %.4f, p-value = %.4f' % (test_stat, pvalue))

# burada kabin numarası 1 olanlar yani kabin numarası olanların toplamını aldım,
# sonra kabin nuaası olmayanlaro topladım, oran testini yapmak için iki parammetre var birincisi başarı sayısı ikincisi gözlem sayısı
# gözlem sayısı oran dolayısıyla kabin numarası olanlar için problem nedir hayatta kalmaktır.nobsta kabin numarası olanlar ve olmayanlar kaç kişi diye sordum
#bunları da nobs argumanına girdim, dikkat propotions z testinin sonuçları şu şekildedir,
# p1 ve p2 oranları arasında fark yoktur der, p1 ve p2 oranları neyi ifade eder? kabin numarası olanlar ve olmayanların hayatta kalıp kalmama durumunu ifade etmektedir.
#ikisi arasında fark yoktur diyen h0 hipotezi p value değeri 0.05ten küçük olduğundan dolayı reddedilir.yani aralarında istatistiki olarak anlamlı bir fark gibi gözüküyor,
# bir değişken türettik, başta benim için anlamlı olur mu olmaz mı bilmiyorum, modelleme yapmadan tam anlamıyla bunu göremem ama bana bir fikir vermesi açısından bu oranlar frekanslar da bunun dağılımı da 
#göz önünde bulundurulduğunda acaba birbirinden farklı mıdırı sordum ve farklıdır yanıtı geldi, dikkat çok değişkenli etkiyi bilmiyorum iki tane değişkeni saedce ikisi birlikte oluşmuş gibi değerlendirerek inceledim
#ama bu yapılar tek başına oluşmadı ki yani survived değişkeninin ortaya çıkışı burada ele aldığımız new cabin bool ile ortaya çıkmadı ki ama çıkmış olabilir bilmiyorum.
#şu andaki kanaatim çıkmış olabileceğine yönelik ama bilmiyorum, dolayısıyla neyi ifade etmeye çalışıyorum, çok değişkenli etkiyi bilmiyorum ben bu değişkenleri birlikte modele soktuğumda çokkdeğişkenli etkiyi gözlemleme imkanım olacak
#o durumda bu değişkenin anlamlı olup olmadığını daha iyi değerlendiriyor olacağım. newcabinbool değişkeninde ama şu anda ilerlemek ya da bu değişkeni kabul etmeye çalışmak için bir fikir edinmeye çalıştığımda 
#yeterli delili buldum. ancak tekrar dikka tçok değişkenli etkiyi bilmiyorum bunu test etmedim. doalyısıyla bu benim için çok önemli değişkendir genellemesini yapamıyorum.

Test Stat = 9.4597, p-value = 0.0000


bir iki tane daha binary feature oluşturuyorum

In [8]:
df.loc[((df['SibSp'] + df['Parch']) > 0), "NEW_IS_ALONE"] = "NO"
df.loc[((df['SibSp'] + df['Parch']) == 0), "NEW_IS_ALONE"] = "YES"
#sibspler yakın ve uzak olabilecek bir takım akrabalıkları ifade etmektedir.
#bunların toplamı eğer 0dan büyükse diyorum yeni bir değişken oluştur ve demek ki bu kişi yalnız değilmiş diyorum no yaz
#tam tersi eğer bunların toplamı 0a eşitse bu kişi yalnız olduğu için yes yazdır
#burada elimde iki adet değişken var bu iki değişkeni bir araya getirdim ve bunlara göre bir koşul oluşturdum.
#bu sayedde olası featureları türetmeye çalışıyorum, belki de kişi gemide tek olup olmamasına göre hayatta kalma refleksleri şekillenmiştir.
#neticesind bir sonuç çıkmayabilir de ancak deniyorum ve bunu suriveda gönderiyorum

In [9]:
df.groupby("NEW_IS_ALONE").agg({"Survived":"mean"})

,Survived
NEW_IS_ALONE,
NO,0.506
YES,0.304


çıkan sonuca göre aralarında bir fark var gibi gözüküyor, yalnız olanların hayatta kalma oranı daha düşük gözüküyor, belki de çok değişkenli etkiden böyle oldu ancak bilmiyorum bir hipotez testi yapacağım. fark var gibi gözküyor ancak istatistiki olarak anlamlı bir fark mı bu diye bakacağım

In [10]:
from statsmodels.stats.proportion import proportions_ztest

test_stat, pvalue = proportions_ztest(count=[df.loc[df["NEW_IS_ALONE"] == "YES", "Survived"].sum(),
                                            df.loc[df["NEW_IS_ALONE"] == "NO", "Survived"].sum()],
                                      
                                      nobs=[df.loc[df["NEW_IS_ALONE"] == "YES", "Survived"].shape[0],
                                           df.loc[df["NEW_IS_ALONE"] == "NO", "Survived"].shape[0]])

print('Test Stat = %.4f, p-value = %.4f' % (test_stat, pvalue))

Test Stat = -6.0704, p-value = 0.0000


evet fark var gibi gözüküyor h0 hipotezi reddedilir. h0 hipotezi der ki iki oran arasında farkı yoktur, tamam reddedildi varmış. bu durum mutlak olarak hayatta kalma durumunu etkilediğini göstermez ancak göz ardı edilemeyeceğini gösterir. bunun etkisini daha net şekilde modelleme bölümünde göreceğiz.

# TEXT FEATURES

metinler üzerinden özellikler türetmeye çalışacağız.

In [12]:
df = load()
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.000,1,0,A/5 21171,7.250,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.000,1,0,PC 17599,71.283,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.000,0,0,STON/O2. 3101282,7.925,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.000,1,0,113803,53.100,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.000,0,0,373450,8.050,NaN,S


In [13]:
# örneğin name değişkenindeki metinlerden değişken türetmeye çalışacağım. bruada 
#fazlaca metin var normalde kardinalitesi yüksek benim içim çöp olan bir değişkendi ancak içerisinden 
#featurelar türetilebilir.


#örneğin harf saydırma, buradaki isimleri saydırarak kaç harf oldukalrını bulabiliriz.
#burada dikkat yorum yapıyorum, çok önemli olmak zorunda değil. incelemeye çalışıyorum emta bilgi geldi aklıma vs.
#örn kraliyet ailesinden biri vrdır belki. sadece deniyorum. yaratıcı olmam gerekiyor

In [14]:
df["NEW_NAME_COUNT"] = df["Name"].str.len()
df
#burada bir değişkendeki harf sayısı ne kadardır bu şekilde hesaplamış olurum.

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,NEW_NAME_COUNT
0,1,0,3,"Braund, Mr. Owen Harris",male,22.000,1,0,A/5 21171,7.250,NaN,S,23
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.000,1,0,PC 17599,71.283,C85,C,51
2,3,1,3,"Heikkinen, Miss. Laina",female,26.000,0,0,STON/O2. 3101282,7.925,NaN,S,22
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.000,1,0,113803,53.100,C123,S,44
4,5,0,3,"Allen, Mr. William Henry",male,35.000,0,0,373450,8.050,NaN,S,24
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.458,NaN,Q,16
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.000,0,0,17463,51.862,E46,S,23
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.000,3,1,349909,21.075,NaN,S,30
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.000,0,2,347742,11.133,NaN,S,49
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.000,1,0,237736,30.071,NaN,C,35


In [15]:
#harfleri saydıktan sonra kelimeleri de sayabilirim 
#word count

df["NEW_NAME_WORD_COUNT"] = df["Name"].apply(lambda x: len(str(x).split(" ")))
# lambda ilgili ismi yakaladığında str'ye çevirsin daha sonra boşluklara göre split etsin
#splitten sonra kaç kelime varsa bunları say ve bunu yeni değişkene ata.
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,NEW_NAME_COUNT,NEW_NAME_WORD_COUNT
0,1,0,3,"Braund, Mr. Owen Harris",male,22.000,1,0,A/5 21171,7.250,NaN,S,23,4
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.000,1,0,PC 17599,71.283,C85,C,51,7
2,3,1,3,"Heikkinen, Miss. Laina",female,26.000,0,0,STON/O2. 3101282,7.925,NaN,S,22,3
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.000,1,0,113803,53.100,C123,S,44,7
4,5,0,3,"Allen, Mr. William Henry",male,35.000,0,0,373450,8.050,NaN,S,24,4
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.458,NaN,Q,16,3
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.000,0,0,17463,51.862,E46,S,23,4
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.000,3,1,349909,21.075,NaN,S,30,4
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.000,0,2,347742,11.133,NaN,S,49,7
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.000,1,0,237736,30.071,NaN,C,35,5


In [16]:
#bu textlerin içerisindeki özel yapıları yakalamaya çalışalım.
# teori bölümünde görmüştük ki bu texlerin içerisinde dr ifadeleri vs vardı bunları yakalamaya çalışacağım.
# örn dr ifadesine sahip olanları flaglesek, benim elimde gemidekilerin meslekleri hakkında bilgi yok ancak
# burada önemli bir bakış açısıyla featurelarımı türeterek değerlendireceğim, modelleme konusunda değerlendiririm diyorum
# ancak burada asıl yapmak istediğim şey olabildiğince az sayıda değişkenle yüksek başarılara erişmektir bunu da unutmamak lazım.
# o yüzden rare encodingte elemeler yaptık. ancak feature eng bölümünde öyle featurelar türeyebilir ki
# hataları düşürebilir, model başarımı yükseltebilir. o yüzden olabildiğince kurcalıyorum.

df["NEW_NAME_DR"] = df["Name"].apply(lambda x: len([x for x in x.split() if x.startswith("Dr")]))
#ilk olarak bu name sütununu split et sonra buradaki her bir kelime liste olarak erişebiliyor olacak
#splitten sonra for ile bu listeleri gez, eğer her listenin başı startswith ile başlıyorsa bunu seç diyorum.
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,NEW_NAME_COUNT,NEW_NAME_WORD_COUNT,NEW_NAME_DR
0,1,0,3,"Braund, Mr. Owen Harris",male,22.000,1,0,A/5 21171,7.250,NaN,S,23,4,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.000,1,0,PC 17599,71.283,C85,C,51,7,0
2,3,1,3,"Heikkinen, Miss. Laina",female,26.000,0,0,STON/O2. 3101282,7.925,NaN,S,22,3,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.000,1,0,113803,53.100,C123,S,44,7,0
4,5,0,3,"Allen, Mr. William Henry",male,35.000,0,0,373450,8.050,NaN,S,24,4,0


In [17]:
#burada göremiyorum tabi, şimdi bunu drye göre groupbya alıp surviveda göre ortalamsını alacağım

df.groupby("NEW_NAME_DR").agg({"Survived": "mean"})
#burada surviveda göre ortalamasını aldıktan sonra dr olanların hayatta kalma oranı daha yüksektir.


,Survived
NEW_NAME_DR,
0,0.383
1,0.500


In [18]:
#kategorik değişken olduğu için frekanslarını gözlemliyoruz.
df.groupby("NEW_NAME_DR").agg({"Survived": ["mean", "count"]})#countunu gözlemliyorum
#900 gözlemin oldupu yerde 1 0tane doktor olması ne kadar önemli tartışmalı olduğu için dursun bi bakalım diyorum.

Survived      
                mean count
NEW_NAME_DR               
0              0.383   881
1              0.500    10

# REGEX FEATURES

In [19]:
#regex ile değişken türeteceğiz.
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,NEW_NAME_COUNT,NEW_NAME_WORD_COUNT,NEW_NAME_DR
0,1,0,3,"Braund, Mr. Owen Harris",male,22.000,1,0,A/5 21171,7.250,NaN,S,23,4,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.000,1,0,PC 17599,71.283,C85,C,51,7,0
2,3,1,3,"Heikkinen, Miss. Laina",female,26.000,0,0,STON/O2. 3101282,7.925,NaN,S,22,3,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.000,1,0,113803,53.100,C123,S,44,7,0
4,5,0,3,"Allen, Mr. William Henry",male,35.000,0,0,373450,8.050,NaN,S,24,4,0


In [20]:
#buradaki titlelar bizim için önemli olabilir name kolonunu kontrol edeceğim.
#burada nasıl bir yol izlemem gerektiğini düşünüyorum. burada bir pattern, örüntü yakalamaya çalışıyorum
#burada regex ile bir feature üretmek istediğimde bir örüntü yakalamam gerekiyor. örn Mrs. için öncesinde ve sonrasında 
#boşluk olup büyük harfle başlayacak ve sonunda nokta olacak. bu diğer titlelar için de geçerli

df["NEW_TITLE"] = df.Name.str.extract(' ([A-Za-z]+)\.', expand=False)
# extract çıkar diyorum, nasıl çıkaracağım, başında boşluk olsun büyük harfle başlayacak
# sonunda nokta olacak, büyük ya da küçük harflerden oluşacak şekilde göreceğin bu harfleri yakala
# her senaryoya göre özel olarak ayarlanması gerekmektedir.
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,NEW_NAME_COUNT,NEW_NAME_WORD_COUNT,NEW_NAME_DR,NEW_TITLE
0,1,0,3,"Braund, Mr. Owen Harris",male,22.000,1,0,A/5 21171,7.250,NaN,S,23,4,0,Mr
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.000,1,0,PC 17599,71.283,C85,C,51,7,0,Mrs
2,3,1,3,"Heikkinen, Miss. Laina",female,26.000,0,0,STON/O2. 3101282,7.925,NaN,S,22,3,0,Miss
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.000,1,0,113803,53.100,C123,S,44,7,0,Mrs
4,5,0,3,"Allen, Mr. William Henry",male,35.000,0,0,373450,8.050,NaN,S,24,4,0,Mr


In [22]:
df[["NEW_TITLE", "Survived","Age"]].groupby(["NEW_TITLE"]).agg({"Survived":"mean", "Age": ["count","mean"]})
# new titleı seç, survivedı seç, agei seç new_titlea göre groupbya al, sonra survivedın ortalamasını al yaş değişkeninin
# countunu ve ortalamsını al


Survived   Age       
              mean count   mean
NEW_TITLE                      
Capt         0.000     1 70.000
Col          0.500     2 58.000
Countess     1.000     1 33.000
Don          0.000     1 40.000
Dr           0.429     6 42.000
Jonkheer     0.000     1 38.000
Lady         1.000     1 48.000
Major        0.500     2 48.500
Master       0.575    36  4.574
Miss         0.698   146 21.774
Mlle         1.000     2 24.000
Mme          1.000     1 24.000
Mr           0.157   398 32.368
Mrs          0.792   108 35.898
Ms           1.000     1 28.000
Rev          0.000     6 43.167
Sir          1.000     1 49.000

bizim için gözlenme ihtimali olmayan değişkenden bile çok değerli bir bilgi çıkardık. titlelarımı çıkardım örn kaptan varmış. drda bulunan frekas farklılığını sonra araştıracağız. buradan çıkardığım bilgiye göre ortık örn yaş değişkeninde tamamen hepsine ortalama ile doldurmamam gerektiğini anladım, neden? titlelara göre yaş ortalamaları zaten farklı çıkıyor. frekasnları yüksek olan titleların yaş ortalaması kayda değer bilgi içeriyor. örn artık yaş değişkenimdeki boşlukları titleların ortalamasına göre doldurmam daha mantıklı. değişken üretmenin olası senaryolarda ne kadar faydalı olabileceğini göstermiş olduk, analiz yaptık.

# DATE FEATURES

bu bölümde date featureları üretiyor olacağız.

In [23]:
dff = pd.read_csv("course_reviews.csv")
dff.head()

,Rating,Timestamp,Enrolled,Progress,Questions Asked,Questions Answered
0,5.000,2021-02-05 07:45:55,2021-01-25 15:12:08,5.000,0.000,0.000
1,5.000,2021-02-04 21:05:32,2021-02-04 20:43:40,1.000,0.000,0.000
2,4.500,2021-02-04 20:34:03,2019-07-04 23:23:27,1.000,0.000,0.000
3,5.000,2021-02-04 16:56:28,2021-02-04 14:41:29,10.000,0.000,0.000
4,4.000,2021-02-04 15:00:24,2020-10-13 03:10:07,10.000,0.000,0.000


In [24]:
dff.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4323 entries, 0 to 4322
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Rating              4323 non-null   float64
 1   Timestamp           4323 non-null   object 
 2   Enrolled            4323 non-null   object 
 3   Progress            4323 non-null   float64
 4   Questions Asked     4323 non-null   float64
 5   Questions Answered  4323 non-null   float64
dtypes: float64(4), object(2)
memory usage: 202.8+ KB


kursa yapılan puanlamalar , puanlama tarihi , kursa üye olma tarihi ilerleme durumu, sorduğu sorular, yanıtlanan sorular gibi değişkenler var, amacım timestamp değişkeninden yeni değişkenler türetmek.

ancak bir problemim var tarih değişkeni olması gereken değişken object tipinde

In [25]:
dff["Timestamp"] = pd.to_datetime(dff["Timestamp"], format = "%Y-%m-%d")
# to_datetime metodu dönüştürmek istediğimiz sütuunu ister ve formatını girmemizi bekler

In [27]:
dff.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4323 entries, 0 to 4322
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Rating              4323 non-null   float64       
 1   Timestamp           4323 non-null   datetime64[ns]
 2   Enrolled            4323 non-null   object        
 3   Progress            4323 non-null   float64       
 4   Questions Asked     4323 non-null   float64       
 5   Questions Answered  4323 non-null   float64       
dtypes: datetime64[ns](1), float64(4), object(1)
memory usage: 202.8+ KB


In [32]:
# örneğin buradan bir yıl değişkeni türetmeye çalışıyorum

dff["year"] = dff["Timestamp"].dt.year #dt datetime modelünü kullanarak yılı çek diyorum
dff.head()

,Rating,Timestamp,Enrolled,Progress,Questions Asked,Questions Answered,year,month
0,5.000,2021-02-05 07:45:55,2021-01-25 15:12:08,5.000,0.000,0.000,2021,2
1,5.000,2021-02-04 21:05:32,2021-02-04 20:43:40,1.000,0.000,0.000,2021,2
2,4.500,2021-02-04 20:34:03,2019-07-04 23:23:27,1.000,0.000,0.000,2021,2
3,5.000,2021-02-04 16:56:28,2021-02-04 14:41:29,10.000,0.000,0.000,2021,2
4,4.000,2021-02-04 15:00:24,2020-10-13 03:10:07,10.000,0.000,0.000,2021,2


In [33]:
dff["month"] = dff["Timestamp"].dt.month # aynı işlem ay için
dff.head()

,Rating,Timestamp,Enrolled,Progress,Questions Asked,Questions Answered,year,month
0,5.000,2021-02-05 07:45:55,2021-01-25 15:12:08,5.000,0.000,0.000,2021,2
1,5.000,2021-02-04 21:05:32,2021-02-04 20:43:40,1.000,0.000,0.000,2021,2
2,4.500,2021-02-04 20:34:03,2019-07-04 23:23:27,1.000,0.000,0.000,2021,2
3,5.000,2021-02-04 16:56:28,2021-02-04 14:41:29,10.000,0.000,0.000,2021,2
4,4.000,2021-02-04 15:00:24,2020-10-13 03:10:07,10.000,0.000,0.000,2021,2


In [34]:
dff['year_diff'] = date.today().year - dff["Timestamp"].dt.year
dff.head() #yılların farkını geetirebilirim bu yıldan

,Rating,Timestamp,Enrolled,Progress,Questions Asked,Questions Answered,year,month,year_diff
0,5.000,2021-02-05 07:45:55,2021-01-25 15:12:08,5.000,0.000,0.000,2021,2,3
1,5.000,2021-02-04 21:05:32,2021-02-04 20:43:40,1.000,0.000,0.000,2021,2,3
2,4.500,2021-02-04 20:34:03,2019-07-04 23:23:27,1.000,0.000,0.000,2021,2,3
3,5.000,2021-02-04 16:56:28,2021-02-04 14:41:29,10.000,0.000,0.000,2021,2,3
4,4.000,2021-02-04 15:00:24,2020-10-13 03:10:07,10.000,0.000,0.000,2021,2,3


In [35]:
# iki tarih arasındaki ay farkı: yıl farkı + ay farkı

dff["month_diff"] = (date.today().year - dff["Timestamp"].dt.year) * 12 + date.today().month - dff["Timestamp"].dt.month
# iki tarih arasındaki ay farkını hesaplamak içinn bunu önce yıl olarak hesaplayıp sonra bunu ay şeklindeçevirmemiz lazım
# sonra bunların ay farkını alıyorum
dff.head()

,Rating,Timestamp,Enrolled,Progress,Questions Asked,Questions Answered,year,month,year_diff,month_diff
0,5.000,2021-02-05 07:45:55,2021-01-25 15:12:08,5.000,0.000,0.000,2021,2,3,42
1,5.000,2021-02-04 21:05:32,2021-02-04 20:43:40,1.000,0.000,0.000,2021,2,3,42
2,4.500,2021-02-04 20:34:03,2019-07-04 23:23:27,1.000,0.000,0.000,2021,2,3,42
3,5.000,2021-02-04 16:56:28,2021-02-04 14:41:29,10.000,0.000,0.000,2021,2,3,42
4,4.000,2021-02-04 15:00:24,2020-10-13 03:10:07,10.000,0.000,0.000,2021,2,3,42


In [36]:
# günlerin isim bilgilerine erişmek istersem

dff["day_name"] = dff["Timestamp"].dt.day_name()
dff.head() # day name metodunu kullanarak bu bilgiye ulaşa bilirim

,Rating,Timestamp,Enrolled,Progress,Questions Asked,Questions Answered,year,month,year_diff,month_diff,day_name
0,5.000,2021-02-05 07:45:55,2021-01-25 15:12:08,5.000,0.000,0.000,2021,2,3,42,Friday
1,5.000,2021-02-04 21:05:32,2021-02-04 20:43:40,1.000,0.000,0.000,2021,2,3,42,Thursday
2,4.500,2021-02-04 20:34:03,2019-07-04 23:23:27,1.000,0.000,0.000,2021,2,3,42,Thursday
3,5.000,2021-02-04 16:56:28,2021-02-04 14:41:29,10.000,0.000,0.000,2021,2,3,42,Thursday
4,4.000,2021-02-04 15:00:24,2020-10-13 03:10:07,10.000,0.000,0.000,2021,2,3,42,Thursday


# ÖZELLİK ETKİLEŞİMLERİ (FEATURE INTERACTION)

özellik etkileşimi değişkenlerin birbiri ile etkileşimine denmektedir. örneğin iki değişkenin çarpılması, toplanması, birdeğişkenin karesinin küpünün alınması gibi değişkenlerle etkileşim kurmak demektir.

In [37]:
df = load()
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.000,1,0,A/5 21171,7.250,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.000,1,0,PC 17599,71.283,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.000,0,0,STON/O2. 3101282,7.925,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.000,1,0,113803,53.100,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.000,0,0,373450,8.050,NaN,S


In [38]:
#örn yaş değişkeni ile pclası çarpıyorum
df["NEW_AGE_PCLASS"] = df["Age"] * df["Pclass"]
# burda dikkat etmem gereken şey örn pclass 1 ile 3 arasında ya yaş değişşkeni de öyle olsun
#burada bunu çarpmam örn refah açısından bir şey ifadeediyor olabilir, çarpılan sonunç 
#örn yaşı büyük olup pclassı 1 olanla yaşı küçük olup pclassı 3 olan biri arasında fark ifade etmektedir
# yani bu çarpımlar bir teoriye karşılık olarak oluşturulmadır. buradan sonuç olarak 
# yaşı küçük olup 1. sınıfta yolculuk edenleri bulabilmektir ya da yaşı büyük yine de 3. sınıfta acaba refah seviyesi mi düşük
# diye bir tez ortaya koymaktır.
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,NEW_AGE_PCLASS
0,1,0,3,"Braund, Mr. Owen Harris",male,22.000,1,0,A/5 21171,7.250,NaN,S,66.000
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.000,1,0,PC 17599,71.283,C85,C,38.000
2,3,1,3,"Heikkinen, Miss. Laina",female,26.000,0,0,STON/O2. 3101282,7.925,NaN,S,78.000
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.000,1,0,113803,53.100,C123,S,35.000
4,5,0,3,"Allen, Mr. William Henry",male,35.000,0,0,373450,8.050,NaN,S,105.000


In [41]:
df["NEW_FAMILY_SIZE"] = df["SibSp"] + df["Parch"] + 1
df.head() #gemideki aile bireyleri sayılarını bulabiliriz, aile sayısı ve artı kendisi şeklinde

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,NEW_AGE_PCLASS,NEW_FAMILY_SIZE
0,1,0,3,"Braund, Mr. Owen Harris",male,22.000,1,0,A/5 21171,7.250,NaN,S,66.000,2
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.000,1,0,PC 17599,71.283,C85,C,38.000,2
2,3,1,3,"Heikkinen, Miss. Laina",female,26.000,0,0,STON/O2. 3101282,7.925,NaN,S,78.000,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.000,1,0,113803,53.100,C123,S,35.000,2
4,5,0,3,"Allen, Mr. William Henry",male,35.000,0,0,373450,8.050,NaN,S,105.000,1


belirli bir kategorik ya da sayısal dğişkenlerin etkieşimlerine de flagler atılabilir. bu noktalara da değişkenler oluşturulabilir. peki ne demek bu, mesela yaşı 21den küçük erkekler, ya da yaşı 50den büyük kadınlar

In [42]:
df.loc[(df["Sex"] == "male") & (df["Age"] <= 21), "NEW_SEX_CAT"] = "youngmale"
# male olanları ve yaşı 21den küçük eşit olanları bu isimde bir değişkene arayıp bu şekilde isimlendir diyorum
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,NEW_AGE_PCLASS,NEW_FAMILY_SIZE,NEW_SEX_CAT
0,1,0,3,"Braund, Mr. Owen Harris",male,22.000,1,0,A/5 21171,7.250,NaN,S,66.000,2,NaN
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.000,1,0,PC 17599,71.283,C85,C,38.000,2,NaN
2,3,1,3,"Heikkinen, Miss. Laina",female,26.000,0,0,STON/O2. 3101282,7.925,NaN,S,78.000,1,NaN
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.000,1,0,113803,53.100,C123,S,35.000,2,NaN
4,5,0,3,"Allen, Mr. William Henry",male,35.000,0,0,373450,8.050,NaN,S,105.000,1,NaN


In [43]:
df.loc[(df["Sex"] == "male") & (df["Age"] > 22) & (df["Age"] <= 50), "NEW_SEX_CAT"] = "maturemale"
#cisiyeti erkek olan ve yaşı 21den büyük ve 50den küçük eşit olanalrı seç ve bu eşkilde isimlendir.

In [44]:
df.loc[(df["Sex"] == "male") & (df["Age"] > 50), "NEW_SEX_CAT"] = "seniormale"
#yaşı 50den büyük olanlara senior de dedim.
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,NEW_AGE_PCLASS,NEW_FAMILY_SIZE,NEW_SEX_CAT
0,1,0,3,"Braund, Mr. Owen Harris",male,22.000,1,0,A/5 21171,7.250,NaN,S,66.000,2,NaN
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.000,1,0,PC 17599,71.283,C85,C,38.000,2,NaN
2,3,1,3,"Heikkinen, Miss. Laina",female,26.000,0,0,STON/O2. 3101282,7.925,NaN,S,78.000,1,NaN
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.000,1,0,113803,53.100,C123,S,35.000,2,NaN
4,5,0,3,"Allen, Mr. William Henry",male,35.000,0,0,373450,8.050,NaN,S,105.000,1,maturemale


aynı işi kadınlar için de yapıyorum

In [45]:
df.groupby("NEW_SEX_CAT")["Survived"].mean()
#NEW_SEX_CAT göre survived değişkenini groupbya alıp ortalamsını alıyorum

NEW_SEX_CAT
maturemale   0.207
seniormale   0.128
youngmale    0.250
Name: Survived, dtype: float64

In [46]:
#olgun kadınları yaptığımı varsaydım ve hayatta kalma oranları 0.74 çıkarken 
#seniormalelerin hayatta kalma oranı sadece 0.128 dir yani bu şekilde mantıklı is
#istatistiksel sonuçlar elde edebilirim.

# 5 ) UYGULAMA

In [47]:
#bu bölümde titanic verisini uçtan uca tamamen ele alacağız, öğrendiğimiz tüm
#metodları uygulayacağız.

df = load()

In [48]:
df.shape

(891, 12)

In [50]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.000,1,0,A/5 21171,7.250,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.000,1,0,PC 17599,71.283,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.000,0,0,STON/O2. 3101282,7.925,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.000,1,0,113803,53.100,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.000,0,0,373450,8.050,NaN,S


In [51]:
#bu veri setinde amaç insanların hayatta kalıp kalamayacağını modellemeye çalışmaktır.
#biz de modelleme öncesi gereken tüm işlemleri yapacağız.

In [52]:
#verideki tüm harfleri büyüteceğiz çünkü tek formatta olmasını istiyorum

df.columns = [col.upper() for col in df.columns] #kolonları gez ve büyült diyorum

In [53]:
#1. adım Feature Engineering (Değişken Mühendisliği)

# yaptıklarım var zaten
#cabinbooldan seniorfemaleye kadar son yaptığım.
#önce aykırı değerler sonra eksik değerleri işledi
#bir değişkendeki eksiklik ondan türetilen değişkenlerde de devam eder.
#artık standartlaştırma bölümünden sonra model aşamasına geçceğiz.

In [54]:
#kalan kısımları sonra yazacağım.